In [3]:
pip install transformers datasets accelerate -q

Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import numpy as np
import pandas as pd
import torch

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from datasets import Dataset

/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
BASE_DIR = ".."
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")

df_train = pd.read_csv(os.path.join(PROCESSED_DIR, "train.csv"))
df_val = pd.read_csv(os.path.join(PROCESSED_DIR, "val.csv"))
df_test = pd.read_csv(os.path.join(PROCESSED_DIR, "test.csv"))

# подгружаем supercategory mapping
mapping = pd.read_csv(os.path.join(PROCESSED_DIR, "label_to_supercategory_v1.csv"))
label_to_supercat = dict(zip(mapping["label"], mapping["supercategory"]))

for df in [df_train, df_val, df_test]:
    df["supercategory"] = df["label"].map(label_to_supercat)

print(df_train["supercategory"].value_counts())

supercategory
sysadmin_devops_network     3308
generic_it_ops              2913
it_governance_leadership    2129
project_product             2006
technical_specialized       1982
backend_general_dev         1546
sales_account               1029
tech_support_helpdesk       1016
web_frontend                 601
Name: count, dtype: int64


In [6]:
le = LabelEncoder()

df_train["y"] = le.fit_transform(df_train["supercategory"])
df_val["y"] = le.transform(df_val["supercategory"])
df_test["y"] = le.transform(df_test["supercategory"])

num_labels = len(le.classes_)
print("Num labels:", num_labels)

Num labels: 9


In [7]:
from transformers import AutoModel

AutoModel.from_pretrained("bert-base-uncased")

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [8]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [9]:
def tokenize(batch):
    return tokenizer(
        batch["resume_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [10]:
train_dataset = Dataset.from_pandas(df_train[["resume_text", "y"]])
val_dataset = Dataset.from_pandas(df_val[["resume_text", "y"]])
test_dataset = Dataset.from_pandas(df_test[["resume_text", "y"]])

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.rename_column("y", "labels")
val_dataset = val_dataset.rename_column("y", "labels")
test_dataset = test_dataset.rename_column("y", "labels")

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map: 100%|██████████| 5510/5510 [00:09<00:00, 605.27 examples/s]


In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
training_args = TrainingArguments(
    output_dir="./bert_9classes",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,   # fast
    weight_decay=0.01,
    logging_steps=100
)

In [13]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro")
    }

In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
100,2.143600
200,2.049000
300,1.865300
400,1.651500
500,1.528400
600,1.421300
700,1.290000
800,1.268100
900,1.318200
1000,1.207000


TrainOutput(global_step=2067, training_loss=1.3619724526518373, metrics={'train_runtime': 2458.3737, 'train_samples_per_second': 6.724, 'train_steps_per_second': 0.841, 'total_flos': 1087374773675520.0, 'train_loss': 1.3619724526518373, 'epoch': 1.0})

In [15]:
predictions = trainer.predict(test_dataset)

y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)

print("Accuracy:", accuracy_score(y_true, y_pred))
print("Macro F1:", f1_score(y_true, y_pred, average="macro"))

/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Accuracy: 0.592377495462795
Macro F1: 0.6084155621359023


In [16]:
# === Reweighting by city_group (train only) ===

# 1) city group counts
city_counts_train = df_train["city_group"].value_counts(dropna=False)

# 2) inverse-frequency weights
# base formula: w_g = 1 / n_g
# then normalize so that the average weight is 1 (more stable optimization)
raw_w = 1.0 / city_counts_train
norm_w = raw_w / raw_w.mean()

city_weight_map = norm_w.to_dict()

# 3) assign each object its group weight
df_train_rw = df_train.copy()
df_val_rw = df_val.copy()
df_test_rw = df_test.copy()

df_train_rw["sample_weight"] = df_train_rw["city_group"].map(city_weight_map).astype(float)

# val/test: weight = 1 (we don't "reweight" there, only evaluate)
df_val_rw["sample_weight"] = 1.0
df_test_rw["sample_weight"] = 1.0

print("Train weights summary:")
display(df_train_rw["sample_weight"].describe())

print("\nTop-10 highest-weight cities (rarest groups):")
display(pd.Series(city_weight_map).sort_values(ascending=False).head(10))

print("\nTop-10 lowest-weight cities (most frequent groups):")
display(pd.Series(city_weight_map).sort_values(ascending=True).head(10))

Train weights summary:


count    16530.000000
mean         0.271415
std          0.444703
min          0.019481
25%          0.019481
50%          0.029519
75%          0.374749
max          1.919766
Name: sample_weight, dtype: float64


Top-10 highest-weight cities (rarest groups):


Набережные Челны    1.919766
Кемерово            1.854690
Сочи                1.793880
Тверь               1.736932
Пенза               1.736932
Оренбург            1.736932
Ставрополь          1.709792
Симферополь         1.633234
Ульяновск           1.541221
Калуга              1.541221
dtype: float64


Top-10 lowest-weight cities (most frequent groups):


Москва             0.019481
Other              0.029519
Санкт-Петербург    0.060894
Краснодар          0.253303
Новосибирск        0.303121
Казань             0.309991
Екатеринбург       0.374749
Самара             0.403789
Ростов-на-Дону     0.450316
Нижний Новгород    0.465645
dtype: float64

In [17]:
# === Build HuggingFace Datasets with sample_weight ===

train_dataset_rw = Dataset.from_pandas(df_train_rw[["resume_text", "y", "sample_weight"]])
val_dataset_rw   = Dataset.from_pandas(df_val_rw[["resume_text", "y", "sample_weight"]])
test_dataset_rw  = Dataset.from_pandas(df_test_rw[["resume_text", "y", "sample_weight"]])

train_dataset_rw = train_dataset_rw.map(tokenize, batched=True)
val_dataset_rw   = val_dataset_rw.map(tokenize, batched=True)
test_dataset_rw  = test_dataset_rw.map(tokenize, batched=True)

train_dataset_rw = train_dataset_rw.rename_column("y", "labels")
val_dataset_rw   = val_dataset_rw.rename_column("y", "labels")
test_dataset_rw  = test_dataset_rw.rename_column("y", "labels")

train_dataset_rw.set_format("torch", columns=["input_ids", "attention_mask", "labels", "sample_weight"])
val_dataset_rw.set_format("torch", columns=["input_ids", "attention_mask", "labels", "sample_weight"])
test_dataset_rw.set_format("torch", columns=["input_ids", "attention_mask", "labels", "sample_weight"])

print(train_dataset_rw[0].keys())

Map: 100%|██████████| 5510/5510 [00:08<00:00, 656.72 examples/s]

dict_keys(['labels', 'sample_weight', 'input_ids', 'attention_mask'])


In [18]:
import torch
from transformers import Trainer

class WeightedTrainer(Trainer):
    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,   # <-- added
        **kwargs                   # <-- for compatibility
    ):
        sample_weight = inputs.pop("sample_weight", None)
        labels = inputs.get("labels")

        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = torch.nn.CrossEntropyLoss(reduction="none")
        per_sample_loss = loss_fct(logits, labels)

        if sample_weight is None:
            loss = per_sample_loss.mean()
        else:
            sample_weight = sample_weight.to(per_sample_loss.dtype)
            loss = (per_sample_loss * sample_weight).mean()

        return (loss, outputs) if return_outputs else loss

In [19]:
# === Reweighted fine-tuning run ===

model_rw = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

trainer_rw = WeightedTrainer(
    model=model_rw,
    args=training_args,          # те же args, 1 эпоха, как baseline
    train_dataset=train_dataset_rw,
    eval_dataset=val_dataset_rw,
    compute_metrics=compute_metrics
)

trainer_rw.train()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
100,2.116300
200,1.989900
300,1.884700
400,1.756700
500,1.648300
600,1.556000
700,1.404300
800,1.385300
900,1.395300
1000,1.279800


TrainOutput(global_step=2067, training_loss=1.4141345898425688, metrics={'train_runtime': 2202.6794, 'train_samples_per_second': 7.504, 'train_steps_per_second': 0.938, 'total_flos': 1087374773675520.0, 'train_loss': 1.4141345898425688, 'epoch': 1.0})

In [20]:
predictions_rw = trainer_rw.predict(test_dataset_rw)

y_true = predictions_rw.label_ids
y_pred = np.argmax(predictions_rw.predictions, axis=1)

print("Reweighted Accuracy:", accuracy_score(y_true, y_pred))
print("Reweighted Macro F1:", f1_score(y_true, y_pred, average="macro"))

/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Reweighted Accuracy: 0.5945553539019963
Reweighted Macro F1: 0.608353781152881


In [21]:
# === GroupDRO preparation ===

# создаём числовые id городов (ТОЛЬКО по train)
city_to_id = {c:i for i,c in enumerate(df_train["city_group"].astype(str).unique())}

df_train_gdro = df_train.copy()
df_val_gdro   = df_val.copy()
df_test_gdro  = df_test.copy()

df_train_gdro["city_id"] = df_train_gdro["city_group"].astype(str).map(city_to_id).astype(int)
df_val_gdro["city_id"]   = df_val_gdro["city_group"].astype(str).map(city_to_id).fillna(-1).astype(int)
df_test_gdro["city_id"]  = df_test_gdro["city_group"].astype(str).map(city_to_id).fillna(-1).astype(int)

print("Number of city groups:", len(city_to_id))

Number of city groups: 41


In [22]:
# === Build Dataset with city_id ===

train_dataset_gdro = Dataset.from_pandas(df_train_gdro[["resume_text","y","city_id"]])
val_dataset_gdro   = Dataset.from_pandas(df_val_gdro[["resume_text","y","city_id"]])
test_dataset_gdro  = Dataset.from_pandas(df_test_gdro[["resume_text","y","city_id"]])

train_dataset_gdro = train_dataset_gdro.map(tokenize, batched=True)
val_dataset_gdro   = val_dataset_gdro.map(tokenize, batched=True)
test_dataset_gdro  = test_dataset_gdro.map(tokenize, batched=True)

train_dataset_gdro = train_dataset_gdro.rename_column("y", "labels")
val_dataset_gdro   = val_dataset_gdro.rename_column("y", "labels")
test_dataset_gdro  = test_dataset_gdro.rename_column("y", "labels")

train_dataset_gdro.set_format("torch", columns=["input_ids","attention_mask","labels","city_id"])
val_dataset_gdro.set_format("torch", columns=["input_ids","attention_mask","labels","city_id"])
test_dataset_gdro.set_format("torch", columns=["input_ids","attention_mask","labels","city_id"])

print(train_dataset_gdro[0].keys())

Map: 100%|██████████| 5510/5510 [00:07<00:00, 716.05 examples/s]

dict_keys(['labels', 'city_id', 'input_ids', 'attention_mask'])


In [23]:
import torch
from transformers import Trainer

class GroupDROTrainer(Trainer):
    def __init__(self, *args, num_groups: int, eta: float = 0.1, **kwargs):
        super().__init__(*args, **kwargs)
        self.num_groups = num_groups
        self.eta = eta
        self.q = torch.ones(num_groups) / num_groups

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        city_id = inputs.pop("city_id")
        labels = inputs.get("labels")

        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = torch.nn.CrossEntropyLoss(reduction="none")
        per_sample_loss = loss_fct(logits, labels)

        group_loss = torch.zeros(self.num_groups, device=per_sample_loss.device)
        group_present = torch.zeros(self.num_groups, device=per_sample_loss.device)

        for g in range(self.num_groups):
            mask = (city_id == g)
            if mask.any():
                group_loss[g] = per_sample_loss[mask].mean()
                group_present[g] = 1.0

        with torch.no_grad():
            q = self.q.to(per_sample_loss.device)
            q = q * torch.exp(self.eta * group_loss * group_present)
            q = q / q.sum()
            self.q = q.detach().cpu()

        q = self.q.to(per_sample_loss.device)
        loss = (q * group_loss).sum()

        return (loss, outputs) if return_outputs else loss

In [24]:
from transformers import TrainingArguments

training_args_gdro = TrainingArguments(
    output_dir="./bert_9classes_gdro",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=100,
    remove_unused_columns=False,   # важно
)

In [25]:
# === Train GroupDRO model ===

num_groups = len(city_to_id)

model_gdro = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

trainer_gdro = GroupDROTrainer(
    model=model_gdro,
    args=training_args_gdro,
    train_dataset=train_dataset_gdro,
    eval_dataset=val_dataset_gdro,
    compute_metrics=compute_metrics,
    num_groups=num_groups,
    eta=0.1
)

trainer_gdro.train()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
100,1.789900
200,1.977300
300,1.821500
400,1.725600
500,1.717100
600,1.419100
700,1.561000
800,1.426800
900,1.533900
1000,1.327300


TrainOutput(global_step=2067, training_loss=1.423555784774507, metrics={'train_runtime': 2040.0649, 'train_samples_per_second': 8.103, 'train_steps_per_second': 1.013, 'total_flos': 1087374773675520.0, 'train_loss': 1.423555784774507, 'epoch': 1.0})

In [26]:
predictions_gdro = trainer_gdro.predict(test_dataset_gdro)

y_true = predictions_gdro.label_ids
y_pred = np.argmax(predictions_gdro.predictions, axis=1)

print("GroupDRO Accuracy:", accuracy_score(y_true, y_pred))
print("GroupDRO Macro F1:", f1_score(y_true, y_pred, average="macro"))

/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


GroupDRO Accuracy: 0.5357531760435572
GroupDRO Macro F1: 0.5410793867387163


In [27]:
# 13 cell: fairness

df_test_eval = df_test.copy()
df_test_eval["y_true"] = y_true
df_test_eval["y_pred"] = y_pred
df_test_eval["correct"] = df_test_eval["y_true"] == df_test_eval["y_pred"]

def gap_by_group(df, col):
    acc = df.groupby(col)["correct"].mean()
    return acc.max() - acc.min()

print("Gender gap:", gap_by_group(df_test_eval, "gender"))
print("City gap:", gap_by_group(df_test_eval, "city_group"))
print("Age gap:", gap_by_group(df_test_eval, "age_group"))

Gender gap: 0.053589309260747175
City gap: 0.6066334991708127
Age gap: 0.19310086239220092


In [28]:
reweighted_snapshot = {
    "acc": accuracy_score(y_true, y_pred),
    "macro_f1": f1_score(y_true, y_pred, average="macro"),
    "gender_gap_acc": gap_by_group(df_test_eval, "gender"),
    "age_gap_acc": gap_by_group(df_test_eval, "age_group"),
    "city_gap_acc": gap_by_group(df_test_eval, "city_group"),
}
reweighted_snapshot

{'acc': 0.5357531760435572,
 'macro_f1': 0.5410793867387163,
 'gender_gap_acc': np.float64(0.053589309260747175),
 'age_gap_acc': np.float64(0.19310086239220092),
 'city_gap_acc': np.float64(0.6066334991708127)}

In [29]:
import numpy as np
import pandas as pd

# making sure there are the needed columns
assert "y_true" in df_test_eval.columns and "y_pred" in df_test_eval.columns

In [30]:
def ovr_rates_by_group(df, group_col, num_classes):
    """
    Computes per-class (one-vs-rest) TPR/FPR for each group.
    Returns:
      tpr: DataFrame [group x class]
      fpr: DataFrame [group x class]
      support_pos: DataFrame [group x class]  (positives count per class)
      support_neg: DataFrame [group x class]  (negatives count per class)
    """
    groups = sorted(df[group_col].dropna().unique())
    tpr = pd.DataFrame(index=groups, columns=range(num_classes), dtype=float)
    fpr = pd.DataFrame(index=groups, columns=range(num_classes), dtype=float)
    support_pos = pd.DataFrame(index=groups, columns=range(num_classes), dtype=int)
    support_neg = pd.DataFrame(index=groups, columns=range(num_classes), dtype=int)

    y_true = df["y_true"].values
    y_pred = df["y_pred"].values

    for g in groups:
        idx = (df[group_col].values == g)
        yt = y_true[idx]
        yp = y_pred[idx]

        for c in range(num_classes):
            # one-vs-rest
            ytc = (yt == c)
            ypc = (yp == c)

            tp = np.sum(ytc & ypc)
            fn = np.sum(ytc & (~ypc))
            fp = np.sum((~ytc) & ypc)
            tn = np.sum((~ytc) & (~ypc))

            pos = tp + fn
            neg = fp + tn

            support_pos.loc[g, c] = int(pos)
            support_neg.loc[g, c] = int(neg)

            # avoid division by zero
            tpr.loc[g, c] = tp / pos if pos > 0 else np.nan
            fpr.loc[g, c] = fp / neg if neg > 0 else np.nan

    return tpr, fpr, support_pos, support_neg


def summarize_gaps(rate_df):
    """
    rate_df: DataFrame [group x class] with rates (TPR or FPR)
    Returns:
      per_class_gap: Series[class] = max(group)-min(group) ignoring NaNs
      worst_gap: float (max over classes)
      macro_gap: float (mean over classes)
    """
    per_class_gap = rate_df.apply(lambda col: np.nanmax(col.values) - np.nanmin(col.values), axis=0)
    worst_gap = float(np.nanmax(per_class_gap.values))
    macro_gap = float(np.nanmean(per_class_gap.values))
    return per_class_gap, worst_gap, macro_gap


num_classes = int(df_test_eval["y_true"].nunique())
print("Detected num_classes:", num_classes)

for group_col in ["gender", "age_group", "city_group"]:
    tpr, fpr, sup_pos, sup_neg = ovr_rates_by_group(df_test_eval, group_col, num_classes)

    tpr_gap_per_class, tpr_worst, tpr_macro = summarize_gaps(tpr)
    fpr_gap_per_class, fpr_worst, fpr_macro = summarize_gaps(fpr)

    print(f"\n=== {group_col} ===")
    print(f"TPR gap (macro over classes): {tpr_macro:.4f}")
    print(f"TPR gap (worst class):        {tpr_worst:.4f}")
    print(f"FPR gap (macro over classes): {fpr_macro:.4f}")
    print(f"FPR gap (worst class):        {fpr_worst:.4f}")

Detected num_classes: 9

=== gender ===
TPR gap (macro over classes): 0.0545
TPR gap (worst class):        0.1288
FPR gap (macro over classes): 0.0110
FPR gap (worst class):        0.0445

=== age_group ===
TPR gap (macro over classes): 0.3320
TPR gap (worst class):        0.8421
FPR gap (macro over classes): 0.1162
FPR gap (worst class):        0.3084

=== city_group ===
TPR gap (macro over classes): 0.9778
TPR gap (worst class):        1.0000
FPR gap (macro over classes): 0.1723
FPR gap (worst class):        0.3529


In [31]:
def js_divergence(p, q, eps=1e-12):
    p = np.asarray(p, dtype=float) + eps
    q = np.asarray(q, dtype=float) + eps
    p = p / p.sum()
    q = q / q.sum()
    m = 0.5 * (p + q)
    kl_pm = np.sum(p * np.log(p / m))
    kl_qm = np.sum(q * np.log(q / m))
    return 0.5 * (kl_pm + kl_qm)

def pred_distribution(df, num_classes):
    counts = df["y_pred"].value_counts().reindex(range(num_classes), fill_value=0).values
    return counts / counts.sum()

num_classes = int(df_test_eval["y_true"].nunique())
overall_dist = pred_distribution(df_test_eval, num_classes)

for group_col in ["gender", "age_group", "city_group"]:
    print(f"\n=== {group_col} prediction distribution shift (JSD) ===")
    for g, gdf in df_test_eval.groupby(group_col):
        d = pred_distribution(gdf, num_classes)
        jsd = js_divergence(d, overall_dist)
        print(f"{g}: JSD={jsd:.4f}, n={len(gdf)}")


=== gender prediction distribution shift (JSD) ===
Female: JSD=0.0127, n=1052
Male: JSD=0.0007, n=4458

=== age_group prediction distribution shift (JSD) ===
18–21: JSD=0.0835, n=82
22–25: JSD=0.0055, n=292
26–35: JSD=0.0006, n=1923
36–50: JSD=0.0081, n=889
50+: JSD=0.0247, n=68
<18: JSD=0.4020, n=3
Unknown: JSD=0.0005, n=2253

=== city_group prediction distribution shift (JSD) ===
Other: JSD=0.0043, n=1312
Алматы: JSD=0.0285, n=67
Барнаул: JSD=0.0322, n=21
Владивосток: JSD=0.0761, n=21
Волгоград: JSD=0.0516, n=25
Воронеж: JSD=0.0284, n=72
Екатеринбург: JSD=0.0187, n=89
Ижевск: JSD=0.0339, n=19
Иркутск: JSD=0.0724, n=33
Казань: JSD=0.0068, n=103
Калининград: JSD=0.0568, n=23
Калуга: JSD=0.0723, n=18
Кемерово: JSD=0.0634, n=22
Краснодар: JSD=0.0111, n=158
Красноярск: JSD=0.0469, n=49
Липецк: JSD=0.0324, n=25
Минск: JSD=0.0307, n=31
Москва: JSD=0.0031, n=1842
Набережные Челны: JSD=0.0335, n=21
Нижний Новгород: JSD=0.0177, n=83
Новосибирск: JSD=0.0074, n=126
Омск: JSD=0.0390, n=45
Оренбу

In [32]:
# group size
city_counts = df_test_eval["city_group"].value_counts()

# take only cities with >= 30 examples
valid_cities = city_counts[city_counts >= 30].index

df_city_robust = df_test_eval[df_test_eval["city_group"].isin(valid_cities)].copy()

print("Original size:", len(df_test_eval))
print("Filtered size:", len(df_city_robust))
print("Remaining cities:", len(valid_cities))

Original size: 5510
Filtered size: 5131
Remaining cities: 24


In [33]:
num_classes = int(df_city_robust["y_true"].nunique())

tpr, fpr, _, _ = ovr_rates_by_group(df_city_robust, "city_group", num_classes)

tpr_gap_per_class, tpr_worst, tpr_macro = summarize_gaps(tpr)
fpr_gap_per_class, fpr_worst, fpr_macro = summarize_gaps(fpr)

print("\n=== City (n >= 30) ===")
print(f"TPR gap (macro over classes): {tpr_macro:.4f}")
print(f"TPR gap (worst class):        {tpr_worst:.4f}")
print(f"FPR gap (macro over classes): {fpr_macro:.4f}")
print(f"FPR gap (worst class):        {fpr_worst:.4f}")


=== City (n >= 30) ===
TPR gap (macro over classes): 0.9037
TPR gap (worst class):        1.0000
FPR gap (macro over classes): 0.1419
FPR gap (worst class):        0.3023


In [34]:
# SAVE ARTIFACTS: old reweight + old GroupDRO

from pathlib import Path
import json
import joblib

REPO_ROOT = Path.cwd()

def save_hf_model(model_obj, tokenizer_obj, le_obj, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    print("Saving to:", out_dir)

    model_obj.save_pretrained(out_dir, safe_serialization=True)
    tokenizer_obj.save_pretrained(out_dir)

    classes = list(le_obj.classes_)
    label2id = {label: int(i) for i, label in enumerate(classes)}
    id2label = {int(i): label for i, label in enumerate(classes)}

    with open(out_dir / "label_mapping.json", "w", encoding="utf-8") as f:
        json.dump({"label2id": label2id, "id2label": id2label}, f, ensure_ascii=False, indent=2)

    joblib.dump(le_obj, out_dir / "label_encoder.joblib")

# 1) old reweight
save_hf_model(
    model_rw, tokenizer, le,
    REPO_ROOT / "models" / "bert_9classes_old_rw_1ep"
)

# 2) old groupdro
save_hf_model(
    model_gdro, tokenizer, le,
    REPO_ROOT / "models" / "bert_9classes_old_gdro_eta0.1_1ep"
)

print("DONE.")

Saving to: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/models/bert_9classes_old_rw_1ep


Saving to: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/models/bert_9classes_old_gdro_eta0.1_1ep
DONE.
